In [7]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import os

# =============================================================================
# 1. Visualization Settings
# =============================================================================
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = ['Arial', 'DejaVu Sans']
plt.rcParams['pdf.fonttype'] = 42
DPI_SETTING = 600

# =============================================================================
# 2. Data Processing and Plotting
# =============================================================================
def load_csv(name):
    if os.path.exists(f'{name}.csv'):
        return pd.read_csv(f'{name}.csv')
    elif os.path.exists(f'S-KUHIMM+KULFFI.xlsx - {name}.csv'):
        return pd.read_csv(f'S-KUHIMM+KULFFI.xlsx - {name}.csv')
    else:
        raise FileNotFoundError(f"Could not find {name}.csv")

df_prop = load_csv('Propionate(mM)')
donor_cols = [c for c in df_prop.columns if c.startswith('HS-')]

def extract_metabolite(df, condition):
    row = df[df['KULFFI'] == condition]
    if row.empty:
        raise ValueError(f"Condition '{condition}' not found.")
    s_vals = row[donor_cols].iloc[0].astype(str).str.replace(',', '', regex=False).str.strip()
    vals = pd.to_numeric(s_vals, errors='coerce')
    return pd.Series(vals, index=donor_cols)

ctrl_prop = extract_metabolite(df_prop, 'Control')

targets = [
    ('Resistant maltodextrin', '3c', 'RMD'),
    ('Resistant starch', '3d', 'RS'),
    ('Mannose', '3e', 'Mannose')
]

for mat_name, fig_num, label_name in targets:
    mat_prop = extract_metabolite(df_prop, mat_name)

    delta_prop = (mat_prop - ctrl_prop).dropna()

    df_waterfall = pd.DataFrame({
        'Donor': delta_prop.index,
        'Delta': delta_prop.values
    })

    df_waterfall = df_waterfall.sort_values('Delta', ascending=False).reset_index(drop=True)

    colors = ['#d62728' if x > 0 else '#1f77b4' for x in df_waterfall['Delta']]

    # =============================================================================
    # 3. Figure Generation
    # =============================================================================
    fig, ax = plt.subplots(figsize=(6, 4))

    ax.bar(df_waterfall.index, df_waterfall['Delta'], color=colors, width=0.8, edgecolor='none')

    ax.set_ylabel(r'$\Delta$ Propionate (mM)', fontsize=14, fontweight='bold')
    ax.set_xlabel('Donors (Ranked)', fontsize=14, fontweight='bold')

    num_donors = len(df_waterfall)
    tick_positions = [0, 9, 19, 29, 39, 49]
    tick_labels = ['1', '10', '20', '30', '40', '50']

    valid_ticks = [p for p in tick_positions if p < num_donors]
    valid_labels = tick_labels[:len(valid_ticks)]

    ax.set_xticks(valid_ticks)
    ax.set_xticklabels(valid_labels, fontsize=12)

    ax.axhline(0, color='black', linewidth=1.5)

    for spine in ['left', 'bottom']:
        ax.spines[spine].set_linewidth(1.5)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

    ax.text(0.95, 0.95, label_name, transform=ax.transAxes,
            fontsize=16, fontweight='bold', va='top', ha='right', color='black')

    plt.tight_layout()
    plt.savefig(f'Figure_{fig_num}.pdf', dpi=DPI_SETTING, bbox_inches='tight')
    plt.close()